# Feedback loop: generate → check → feed back → retry

A `RefinementLoop` generates an answer, a `QualityCheck` judges it, and if it falls short the
critique is fed back to the generator for another attempt — up to a budget. It returns the accepted
answer or the best-scoring attempt.

Three pluggable checks ship: `KeywordQualityCheck` (deterministic, zero-infra), `ClassifierQualityCheck`
(lexicon/DJL classifier, zero-infra), and `LlmCriticQualityCheck` (a second LLM scores + critiques).
The checks run offline below; the full loop needs a generator LLM, so it's gated on `ANTHROPIC_API_KEY`.

Prereqs: `mvn -DskipTests package`, `pip install -e python/`.

In [1]:
import os, pathlib, subprocess
repo_root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
if 'AGENTIC_FLINK_JAR' not in os.environ:
    jars = [j for j in sorted((repo_root/'target').glob('agentic-flink-*.jar'))
            if 'original' not in j.name and 'sources' not in j.name]
    os.environ['AGENTIC_FLINK_JAR'] = str(jars[-1])
cp = repo_root/'target'/'runtime-classpath.txt'
if not cp.exists():
    subprocess.run(['mvn','-q','dependency:build-classpath',f'-Dmdep.outputFile={cp}',
                    '-Dmdep.includeScope=runtime'], cwd=repo_root, check=True)
import agentic_flink as af
af.start_jvm(extra_jars=[j for j in cp.read_text().strip().split(':') if j])
from agentic_flink import jclass
Keyword = jclass('org.agentic.flink.feedback.KeywordQualityCheck')
ClassifierCheck = jclass('org.agentic.flink.feedback.ClassifierQualityCheck')
RefinementLoop = jclass('org.agentic.flink.feedback.RefinementLoop')
Lexicon = jclass('org.agentic.flink.inference.LexiconInferenceConnection')
InferenceSetup = jclass('org.agentic.flink.inference.InferenceSetup')
key = os.environ.get('ANTHROPIC_API_KEY')
print('ready —', 'generator: Claude' if key else 'generator disabled (no ANTHROPIC_API_KEY)')

ready — generator disabled (no ANTHROPIC_API_KEY)


## The checks, offline
Each returns a 0-1 score, pass/fail, and a critique that would be fed back.

In [2]:
kw = Keyword.requiring('flink', 'streaming')
for ans in ['a vague note', 'Flink is a streaming engine']:
    r = kw.check('Explain Flink', ans)
    print(f'  keyword   passed={r.passed!s:<5} score={r.score:.2f} critique={r.critique} | {ans!r}')

setup = InferenceSetup.builder().withModelName('lexicon').withModelUri('lexicon://x').build()
clf = ClassifierCheck(Lexicon(), setup, 0.3, True)  # pass if suspicion <= 0.3
for ans in ['lunch plans for friday', 'urgent wire transfer gift card verify your account']:
    r = clf.check('write a note', ans)
    print(f'  classifier passed={r.passed!s:<5} score={r.score:.2f} critique={r.critique} | {ans!r}')

  keyword   passed=False score=0.00 critique=missing required terms: [flink, streaming] | 'a vague note'
  keyword   passed=True  score=1.00 critique=None | 'Flink is a streaming engine'
  classifier passed=True  score=1.00 critique=None | 'lunch plans for friday'
  classifier passed=False score=0.07 critique=classifier label 'SUSPICIOUS' score 0.93 fails ceiling 0.30 | 'urgent wire transfer gift card verify your account'


## The full loop (needs ANTHROPIC_API_KEY)

Claude generates, the keyword check gates, and any shortfall is fed back. Prints the per-attempt trace.

In [3]:
if not key:
    print('⚠ No ANTHROPIC_API_KEY — skipping the live loop. The checks above ran offline.')
    print('  Set the key and re-run cell 1 to enable Claude generation + refinement.')
else:
    from java.util import ArrayList
    terms = ArrayList(); [terms.add(t) for t in ['stream', 'state', 'checkpoint']]
    loop = (RefinementLoop.builder()
              .withClaude(key)
              .withCheck(Keyword(terms, 60, 1.0))   # must mention all three + be >=60 chars
              .withMaxAttempts(3)
              .build())
    result = loop.refine('Explain in 2 sentences what Apache Flink is and why checkpoints matter.')
    print('accepted:', result.accepted, '| attempts:', result.attemptsUsed, '| score:', round(result.finalScore(), 2))
    print('\nfinal answer:\n', result.finalText)
    print('\ntrace:')
    for a in result.trace:
        print(f'  attempt {a.index}: score={a.score:.2f} critique={a.critique}')

⚠ No ANTHROPIC_API_KEY — skipping the live loop. The checks above ran offline.
  Set the key and re-run cell 1 to enable Claude generation + refinement.


## Notes

The loop stops on the first passing attempt or when the budget is spent (returning the
highest-scoring attempt). Swap `Keyword(...)` for `ClassifierQualityCheck` or `LlmCriticQualityCheck.claude(key, 0.8, rubric)`
to gate on an ML score or an LLM critic instead. The streaming version is `SelfRefinementExample`.